# DLGenAI Project Milestone-4

#### This milestone focuses on formulating the Smart MCQ Solver Challenge as a proper multiple-choice classification problem. You will learn how to convert each prompt and its five options into model-ready inputs, use AutoModelForMultipleChoice to produce logits for A-E, apply LoRA for efficient fine-tuning, and run a small Hugging Face Trainer fine-tuning pipeline.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.preprocessing import LabelEncoder

W0715 15:05:04.465000 4833 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0715 15:05:04.618000 4833 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Multiple-Choice Data Formatting
In this section, you will convert the Kaggle MCQ format into the structure required by multiple-choice models. Each question has one prompt and five options and each option must be paired with the prompt separately.

Q1. Label Encoding
Convert the answer column in train.csv into numeric labels using the following mapping:
A = 0
B = 1
C = 2
D = 3
E = 4

What is the encoded numeric label for the row at index 150?


In [2]:
df = pd.read_csv('train.csv')

le = LabelEncoder()
le.fit(['A', 'B', 'C', 'D', 'E'])

df['answer'] = le.transform(df['answer'])
df['answer'][150]

np.int64(2)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   int64 
dtypes: int64(2), object(6)
memory usage: 125.1+ KB


Q2. Prompt-Option Formatting
For row index 0, create the Option B input using exactly this format:
str(prompt) + " [SEP] " + str(option_B)

What is the exact character length of this formatted input string?


In [4]:
len(str(df['prompt'][0] + ' [SEP] ' +  df['B'][0]))

407

Tokenization for Multiple-Choice Models
Multiple-choice models expect inputs in the shape:
batch_size x num_choices x sequence_length

Since each question has five options, every row becomes five tokenized sequences.


In [5]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
options = ['A', 'B', 'C', 'D', 'E']

def format(row):
    return [str(row['prompt']) + ' [SEP] ' + str(row[opt]) for opt in options]

Q3. Single-Row MCQ Tokenization
Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
padding = "max_length"
truncation = True
max_length = 128
return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

What is the value of the second dimension?


In [6]:
row0 = df.loc[0]
row0_texts = format(row0)
enc = tokenizer(row0_texts, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
input_ids = enc['input_ids'].unsqueeze(0)
input_ids.shape[1]

5

Q4. Batch MCQ Tokenization
Tokenize the first 16 rows of train.csv as multiple-choice examples.
Each row has 5 choices.
Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?


In [7]:
batch = []

for i in range(16):
    batch.extend(format(df.loc[i]))

enc = tokenizer(batch, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
batch_input_ids = enc['input_ids'].view(16, 5, 128)
batch_input_ids.numel()

10240

Multiple-Choice Model Outputs
AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E.


Q5. Multiple-Choice Logits
Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question?

In [ ]:
model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')

In [9]:
def prepare_row(row):
    texts = format(row)
    enc = tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
    return {'input_ids': enc['input_ids'].unsqueeze(0), 'attention_mask': enc['attention_mask'].unsqueeze(0)}

inputs_row0 = prepare_row(row0)
with torch.no_grad():
    outputs = model(**inputs_row0)
outputs.logits.shape[0] * outputs.logits.shape[1]


5

Q6. Supervised Loss Tensor
For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?


In [10]:
label_tensor = torch.tensor([row0['answer']])
outputs_with_loss = model(**inputs_row0, labels=label_tensor)
outputs_with_loss.loss.dim()


0

LoRA for Efficient Fine-Tuning
LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.


Q7. LoRA Trainable Parameters
Apply LoRA to the bert-base-uncased multiple-choice model using:
r = 8
lora_alpha = 16
target_modules = ["query", "value"]
lora_dropout = 0.1
bias = "none"
task_type = TaskType.SEQ_CLS

Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?


In [ ]:
!pip install --upgrade torch torchvision torchao

In [12]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['query', 'value'],
    lora_dropout=0.1,
    bias='none',
    task_type=TaskType.SEQ_CLS,
)
lora_model = get_peft_model(model, lora_config)
trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
trainable_params


295681

Preparing Data for Hugging Face Trainer
Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.


Q8. Hugging Face Dataset Preparation
Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:
input_ids with shape [5, 128]
attention_mask with shape [5, 128]
labels as the encoded answer label

For the first dataset item, input_ids has shape:
[5, 128]

How many tokenized choices are stored in input_ids?


In [13]:
def hugging_face(row):
    texts = format(row)
    enc = tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
    return {
        'input_ids': enc['input_ids'],
        'attention_mask': enc['attention_mask'],
        'labels': row['answer']
    }

hundred = df.iloc[:100]
examples = [hugging_face(hundred.iloc[i]) for i in range(len(hundred))]
hf_dataset = Dataset.from_list([
    {
        'input_ids': ex['input_ids'].tolist(),
        'attention_mask': ex['attention_mask'].tolist(),
        'labels': ex['labels'],
    } for ex in examples
])

first_item = torch.tensor(hf_dataset[0]['input_ids']).shape
first_item[0]


5

Tiny Fine-Tuning and Inference
In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.


Q9. Tiny LoRA Fine-Tuning
Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:
max_length = 64
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
max_steps = 4

What is the final global_step reported by the Trainer?


In [14]:
def lora_finetuning(row, max_length=64):
    texts = format(row)
    enc = tokenizer(texts, padding='max_length', truncation=True, max_length=max_length, return_tensors='pt')
    return {
        'input_ids': enc['input_ids'].tolist(),
        'attention_mask': enc['attention_mask'].tolist(),
        'labels': row['answer'],
    }

first_32 = df.iloc[:32]
train_examples = [lora_finetuning(first_32.iloc[i]) for i in range(len(first_32))]
train_dataset = Dataset.from_list(train_examples)

def collate_fn(batch):
    return {
        'input_ids': torch.tensor([b['input_ids'] for b in batch]),
        'attention_mask': torch.tensor([b['attention_mask'] for b in batch]),
        'labels': torch.tensor([b['labels'] for b in batch]),
    }

training_args = TrainingArguments(
    output_dir='./mc_lora_out',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to=[],
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
)
train_result = trainer.train()
trainer.state.global_step


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,1.583353
2,1.612520
3,1.508855
4,1.593871


4

Q10. Probability Assigned to Option E After Fine-Tuning
Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places.


In [16]:
inputs_row0_64 = {
    'input_ids': torch.tensor(lora_finetuning(row0)['input_ids']).unsqueeze(0),
    'attention_mask': torch.tensor(lora_finetuning(row0)['attention_mask']).unsqueeze(0),
}
lora_model.eval()
with torch.no_grad():
    out = lora_model(**inputs_row0_64)

probs = F.softmax(out.logits, dim=-1)
prob_e = probs[0][4].item()
prob_e


0.19532330334186554